In [1]:
import pandas as pd

# --- CONFIGURATION ---
# Replace this with your exact Excel filename if different
file_name = 'Research-Questionnaire-Responses.xlsx' 
# ---------------------

print("Loading data...")

# 1. Load Data using read_excel
try:
    df = pd.read_excel(file_name, engine='openpyxl')
except FileNotFoundError:
    print(f"ERROR: The file '{file_name}' was not found.")
    print("Please make sure the Excel file is in the same folder as this script.")
    exit()
except Exception as e:
    print(f"An error occurred while loading: {e}")
    exit()

# Clean column headers (remove accidental spaces)
df.columns = df.columns.str.strip()

# 2. Filter Eligibility
eligible_col = 'Do you meet all of the eligibility criteria to participate in this survey?'

if eligible_col not in df.columns:
    # Try to find the column if the name isn't exact
    possible_cols = [c for c in df.columns if 'eligibility criteria' in str(c).lower()]
    if possible_cols:
        eligible_col = possible_cols[0]
    else:
        print(f"Warning: Eligibility column not found. Proceeding without filtering.")

# Apply Filter if column exists
if eligible_col in df.columns:
    df = df[df[eligible_col].astype(str).str.startswith('Yes')].copy()
    print(f"Filtered eligible participants. Count: {len(df)}")

# ==============================================================================
# 2.5 STATISTICAL PRUNING (Reduce to 385)
# ==============================================================================
target_n = 385
current_n = len(df)

if current_n > target_n:
    # We use .sample() to randomly select rows. 
    # random_state=42 acts as a 'seed' to ensure you get the EXACT same 385 rows 
    # every time you run this script (reproducibility).
    df = df.sample(n=target_n, random_state=42).reset_index(drop=True)
    print(f"--- PRUNING APPLIED ---")
    print(f"Dataset randomly reduced from {current_n} to {target_n} participants.")
else:
    print(f"Pruning not needed. Current count ({current_n}) is already <= {target_n}.")
# ==============================================================================

# 3. Rename Columns
new_cols = ['Age', 'Gender', 'CivilStatus', 'Nationality', 'Education', 
            'StayLength', 'Industry', 'Tenure', 'Role', 'Income']

# Generate scale item names safely
scale_cols = (
    [f'POS_Recog_{i}' for i in range(1,6)] +
    [f'POS_Sup_{i}'   for i in range(1,6)] +
    [f'POS_Cond_{i}'  for i in range(1,6)] +
    [f'POS_Fair_{i}'  for i in range(1,6)] +
    [f'ES_Int_{i}'    for i in range(1,6)] +
    [f'ES_Ext_{i}'    for i in range(1,6)]
)

# Apply slicing (Selecting columns by position)
try:
    # Adjust indices if your Excel structure is different. 
    # Assumes: Col 0-1 are junk/metadata, Col 2-11 are Demo, Col 12-41 are Scales.
    df_demo = df.iloc[:, 2:12]    # 10 Demographic columns
    df_scales = df.iloc[:, 12:42] # 30 Scale columns
    
    # Combine
    df_clean = pd.concat([df_demo, df_scales], axis=1)
    
    # Assign new names
    expected_cols = len(new_cols) + len(scale_cols)
    if len(df_clean.columns) == expected_cols:
        df_clean.columns = new_cols + scale_cols
    else:
        print(f"Error: Column count mismatch. Selected {len(df_clean.columns)} columns, expected {expected_cols}.")
        print("Please check your Excel file structure (columns might have shifted).")
        exit()

except Exception as e:
    print(f"Error during column slicing: {e}")
    exit()

# 4. Convert Text to Numbers
likert_map = {
    'strongly disagree': 1, 
    'disagree': 2, 
    'neutral': 3, 
    'agree': 4, 
    'strongly agree': 5
}

for col in scale_cols:
    # Convert to string, lowercase, strip spaces, then map
    df_clean[col] = df_clean[col].astype(str).str.lower().str.strip().map(likert_map)

# 5. Calculate Means
# Indices in df_clean: 0-9 are Demographics. 10-29 are POS (20 items). 30-39 are ES (10 items).
df_clean['POS_Total_Mean'] = df_clean.iloc[:, 10:30].mean(axis=1)
df_clean['ES_Total_Mean'] = df_clean.iloc[:, 30:40].mean(axis=1)

# 6. Create Groups for ANOVA (Low/Moderate/High)
# Note: We calculate quartiles AFTER pruning so the groups fit the N=385 dataset.
p33 = df_clean['POS_Total_Mean'].quantile(0.33)
p66 = df_clean['POS_Total_Mean'].quantile(0.66)

def classify(x):
    if pd.isna(x): return 'Unknown'
    if x <= p33: return 'Low Support'
    elif x <= p66: return 'Moderate Support'
    else: return 'High Support'

df_clean['POS_Group'] = df_clean['POS_Total_Mean'].apply(classify)

# 7. Save to CSV for Jamovi
output_file = 'Cleaned_Data_Jamovi_385.csv'
df_clean.to_csv(output_file, index=False)
print(f"Success! Data processed and pruned. File saved as: {output_file}")

Loading data...
Filtered eligible participants. Count: 390
--- PRUNING APPLIED ---
Dataset randomly reduced from 390 to 385 participants.
Success! Data processed and pruned. File saved as: Cleaned_Data_Jamovi_385.csv
